In [ ]:
# this notebook is greatly insipred by original RWKV package https://github.com/BlinkDL/ChatRWKV/blob/main/rwkv_pip_package/README.md
# as well as https://huggingface.co/spaces/RWKV-Red-Team/RWKV-LatestSpace/tree/main

In [1]:
!python --version

Python 3.11.13


In [2]:
!pip install pydantic==2.12.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 18.2 MB/s eta 0:00:00
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.4
    Uninstalling pydantic-2.12.4:
      Successfully uninstalled pydantic-2.12.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [5]:
!pip install -i https://test.pypi.org/simple/ rwkvx==0.1.0.dev2

Looking in indexes: https://test.pypi.org/simple/


In [7]:
from rwkvx.conversation import *

In [8]:
templates = {
    'system': FormatPromptTemplate('System: {text}\n\n'),
    'user': FormatPromptTemplate('User: {text}{think_mode}\n\n'),
    'assistant': FormatPromptTemplate('Assistant: {think_start}{think_text}{think_end}{text}'),
}

history = HistoryManager()

mgr = ConversationManager(
    templates=templates,
    history=history,
)

mgr.add_user('Tell me a joke.', metadata={'think_mode': ' think'})

think_prompt = mgr.build_rwkv7_prompt()

assert think_prompt.startswith('User: Tell me a joke. think\n\n')
assert '<think' in think_prompt
assert '</think>' not in think_prompt

history.clear()

mgr.add_user('Tell me another joke.')

nothink_prompt = mgr.build_rwkv7_prompt(user_assistant_count=1)

assert nothink_prompt.startswith('User: Tell me another joke.\n\n')
assert '<think>\n</think>' in nothink_prompt

In [9]:
think_prompt

'User: Tell me a joke. think\n\nAssistant: <think'

In [10]:
nothink_prompt

'User: Tell me another joke.\n\nAssistant: <think>\n</think>'